# C02. 시그모이드 함수

> 📌 **이 모듈에서 배울 것**
> 
> 분류 문제에서 출력을 **0~1 사이 확률**로 변환하는 시그모이드 함수.

## 사전 지식

- C01 (분류 vs 회귀) 완료

---


## 1. 시그모이드 함수란

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

어떤 실수 $z$를 받아서 **0~1 사이 값**으로 변환해주는 함수.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 다양한 입력 → 출력
test_values = [-10, -3, -1, 0, 1, 3, 10]
for z in test_values:
    print(f"σ({z:>4}) = {sigmoid(z):.4f}")


**핵심 특성**:
- `z = 0` → `σ(0) = 0.5` (정확히 가운데)
- `z = +∞` → `σ ≈ 1` (큰 양수는 1에 가까이)
- `z = -∞` → `σ ≈ 0` (큰 음수는 0에 가까이)


In [ ]:
# 그래프
z = np.linspace(-10, 10, 200)
plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid(z), linewidth=2)
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='σ = 0.5 (z=0)')
plt.axhline(0, color='gray', linewidth=0.5)
plt.axhline(1, color='gray', linewidth=0.5)
plt.xlabel('z (model output)')
plt.ylabel('σ(z) (probability)')
plt.title('Sigmoid Function')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 2. 왜 시그모이드를 쓰나?

회귀 모델의 출력 `wx + b`는 어떤 실수든 가능 (-100, 0.7, 500 등).  
이걸 **확률로 해석**하려면 0~1 범위로 변환해야 함.

```
w·x + b → σ(w·x + b) → 0~1 사이 확률
```

이 출력을 "Gentoo일 확률"로 해석할 수 있어요.


In [ ]:
# C01에서 본 회귀 출력 → 시그모이드 적용
np.random.seed(42)
# 시뮬레이션: 모델 출력 (선형) 만들기
linear_outputs = np.random.normal(0, 3, 100)
print(f"선형 출력 범위: {linear_outputs.min():.2f} ~ {linear_outputs.max():.2f}")

# 시그모이드 통과
probabilities = sigmoid(linear_outputs)
print(f"시그모이드 후 범위: {probabilities.min():.4f} ~ {probabilities.max():.4f}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(linear_outputs, bins=20, edgecolor='black')
axes[0].set_title('Before sigmoid (any real number)')
axes[0].set_xlabel('w·x + b')
axes[0].grid(alpha=0.3)

axes[1].hist(probabilities, bins=20, edgecolor='black')
axes[1].set_title('After sigmoid (0~1)')
axes[1].set_xlabel('σ(w·x + b)')
axes[1].set_xlim(0, 1)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 3. PyTorch에서의 시그모이드

PyTorch에는 `torch.sigmoid()` 함수가 있어요. 한 줄로 사용 가능.


In [ ]:
import torch

z = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
sig = torch.sigmoid(z)
print(f"z   = {z}")
print(f"σ(z) = {sig}")


## 4. 도함수 (참고)

시그모이드의 미분은 **자기 자신으로 표현**되는 깔끔한 형태:

$$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$

수학적 유도는 정공법이라 이 보충 트랙에선 다루지 않지만, 이 사실이 cross-entropy 편미분을 깔끔하게 만들어줘요. (다음 모듈에서 결과만 활용)


In [ ]:
# 도함수 시각화 (참고)
z = np.linspace(-6, 6, 200)
sig = sigmoid(z)
sig_grad = sig * (1 - sig)

plt.figure(figsize=(8, 4))
plt.plot(z, sig, label='σ(z)')
plt.plot(z, sig_grad, label="σ'(z) = σ(z)(1-σ(z))", linestyle='--')
plt.xlabel('z')
plt.title('Sigmoid and its derivative')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**도함수 모양 관찰**:
- 종 모양 (z=0 근처에서 최대)
- 최댓값 0.25 (z=0에서)
- 양 끝에서는 거의 0 → "이미 확실한 곳에선 기울기 약함" (학습 천천히)


## 5. ⚠️ 함정 / 주의사항

### 5.1 큰 값에서 포화
`z = 100` 같이 크면 시그모이드가 1에 매우 가까움. 이때 도함수가 거의 0 → **기울기 소실**.  
→ 깊은 신경망에서 문제, 그래서 요즘은 ReLU 등 다른 활성화 함수 사용.

### 5.2 수치 안정성
`exp(-z)`에서 z가 매우 작으면 (예: -1000) overflow. PyTorch의 `torch.sigmoid`는 안전하게 구현돼 있음.

### 5.3 직접 구현 시
```python
def sigmoid(z):
    return 1 / (1 + torch.exp(-z))  # 가능은 함, 하지만 torch.sigmoid가 안전
```


## 6. 📚 더 알아보기

- **Softmax** — 다중 분류용 시그모이드의 일반화
- **Logit** — 시그모이드의 역함수: `log(p / (1-p))`
- **ReLU**, **Tanh** 등 다른 활성화 함수
